# Gradient Field CNN Baseline

Build a compact CNN that ingests gradient field representations and outputs:
1. A binary logit (real vs AI)
2. A 128-D embedding for future fusion

12-channel gradient field (2 scales x 6) -> Gx, Gy, magnitude, angle, coherence

### Mathematical Foundation
- **Luminance**: BT.709 (L = 0.2126R + 0.7152G + 0.0722B)
- **Gradients**: Sobel operators Gx, Gy
- **Magnitude**: |G| = √(Gx² + Gy²)
- **Angle**: θ = atan2(Gy, Gx)
- **Coherence**: κ = ((λ₁ - λ₂) / (λ₁ + λ₂))² from structure tensor

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GDRIVE_DATA_DIR = "/content/drive/MyDrive/datasets"

In [2]:
import os
import time
import math
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix,
    precision_score, recall_score, roc_curve, precision_recall_curve,
    average_precision_score, classification_report
)
import numpy as np
from PIL import Image

""""
if not torch.cuda.is_available():
    raise RuntimeError(
        '❌ CUDA not available! Training on CPU takes ~50 min/epoch.\n'
        'Go to Runtime → Change runtime type → Select GPU (T4 recommended)'
    )
print(f'✅ GPU available: {torch.cuda.get_device_name(0)}')
print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
"""

# Where to save best checkpoints & logs
ARTIFACTS_DIR = Path("models")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
# data_dir = f"{GDRIVE_DATA_DIR}/OpenFake"
epochs = 30
batch_size = 64
lr = 1e-3
device = "cuda" if torch.cuda.is_available() else "cpu"

# DataLoader config optimized for Colab
NUM_WORKERS = 2  # Colab max recommended
PIN_MEMORY = True  # Fast CPU→GPU transfer


# Early Stopping parameters
PATIENCE = 3
patience_counter = 0



print(f"Using device: {device}")

Using device: cpu


In [3]:
PREPROCESSING_CONFIG = {
    # Luminance conversion (BT.709)
    "luminance_coefficients": {"R": 0.2126, "G": 0.7152, "B": 0.0722},

    # Image preprocessing
    "image_size": 224,

    # Gradient computation
    "gradient_operator": "Sobel 3x3",
    "sobel_x": [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
    "sobel_y": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],

    # Coherence (structure tensor)
    "gaussian_kernel_size": 5,
    "gaussian_sigma": "binomial approximation (σ≈1)",

    # Feature computation
    "features": ["Gx", "Gy", "log1p(magnitude*10)", "sin(angle)", "cos(angle)", "coherence²"],
    "scales": ["sigma=1 (fine)", "sigma=3 (coarse)"],
    "channels": 12,
    "epsilon": 1e-8,  # Numerical stability
    "magnitude_scaling": "log1p(mag * 10)",  # Preserves dynamic range
    "angle_normalization": "[-1, 1] via /π",
}

# Save with model checkpoint
import json
with open(ARTIFACTS_DIR / "preprocessing_config_v3.json", "w") as f:
    json.dump(PREPROCESSING_CONFIG, f, indent=2)

print("Preprocessing Parameters:")
for k, v in PREPROCESSING_CONFIG.items():
    print(f"   {k}: {v}")

Preprocessing Parameters:
   luminance_coefficients: {'R': 0.2126, 'G': 0.7152, 'B': 0.0722}
   image_size: 256
   gradient_operator: Sobel 3x3
   sobel_x: [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]
   sobel_y: [[-1, -2, -1], [0, 0, 0], [1, 2, 1]]
   gaussian_kernel_size: 5
   gaussian_sigma: binomial approximation (σ≈1)
   features: ['Gx', 'Gy', 'log1p(magnitude*10)', 'angle/π', 'coherence²']
   epsilon: 1e-08
   magnitude_scaling: log1p(mag * 10)
   angle_normalization: [-1, 1] via /π


---
## 1. Metrics Dataclass

Clean, structured container for all evaluation metrics.

In [4]:
@dataclass
class Metrics:
    """Container for all classification metrics."""
    accuracy: float = 0.0
    precision: float = 0.0
    recall: float = 0.0
    f1: float = 0.0
    specificity: float = 0.0
    auroc: float = 0.0
    avg_precision: float = 0.0  # Area under PR curve

    # Raw data for curve plotting
    labels: np.ndarray = field(default_factory=lambda: np.array([]))
    preds: np.ndarray = field(default_factory=lambda: np.array([]))
    probs: np.ndarray = field(default_factory=lambda: np.array([]))

    def __repr__(self):
        return (
            f"Metrics(acc={self.accuracy:.4f}, prec={self.precision:.4f}, "
            f"rec={self.recall:.4f}, f1={self.f1:.4f}, spec={self.specificity:.4f}, "
            f"auroc={self.auroc:.4f}, ap={self.avg_precision:.4f})"
        )

    def to_dict(self):
        """Return scalar metrics as dict (excludes arrays)."""
        return {
            'accuracy': self.accuracy,
            'precision': self.precision,
            'recall': self.recall,
            'f1': self.f1,
            'specificity': self.specificity,
            'auroc': self.auroc,
            'avg_precision': self.avg_precision
        }


def compute_metrics(labels: np.ndarray, preds: np.ndarray, probs: np.ndarray) -> Metrics:
    """
    Compute all classification metrics robustly.

    Args:
        labels: Ground truth (0=real, 1=fake)
        preds: Binary predictions
        probs: Probability scores for positive class (fake)

    Returns:
        Metrics dataclass with all computed values
    """
    metrics = Metrics(labels=labels, preds=preds, probs=probs)

    if len(labels) == 0:
        return metrics

    # Basic metrics with zero_division handling
    metrics.accuracy = accuracy_score(labels, preds)
    metrics.precision = precision_score(labels, preds, zero_division=0)
    metrics.recall = recall_score(labels, preds, zero_division=0)
    metrics.f1 = f1_score(labels, preds, zero_division=0)

    # Specificity = TN / (TN + FP)
    cm = confusion_matrix(labels, preds)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        metrics.specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # AUROC and Average Precision (require both classes present)
    if len(np.unique(labels)) > 1:
        try:
            metrics.auroc = roc_auc_score(labels, probs)
            metrics.avg_precision = average_precision_score(labels, probs)
        except ValueError:
            metrics.auroc = float('nan')
            metrics.avg_precision = float('nan')
    else:
        metrics.auroc = float('nan')
        metrics.avg_precision = float('nan')

    return metrics

---
## 2. Luminance Dataset

In [5]:
class LuminanceDataset(Dataset):
    """
    GPU-optimized dataset that only loads images and converts to luminance.
    Gradient computation is done on GPU inside the model.
    """
    def __init__(self, img_paths: list[str], labels: list[int], img_size=224):
        self.img_paths = img_paths
        self.labels = labels
        self.img_size = img_size
        self.resize = transforms.Resize((img_size, img_size))

        # BT.709 luminance coefficients
        self.R_COEFF = 0.2126
        self.G_COEFF = 0.7152
        self.B_COEFF = 0.0722

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        img = self.resize(img)
        img_tensor = transforms.functional.to_tensor(img)

        # RGB -> Luminance (BT.709)
        luminance = (self.R_COEFF * img_tensor[0] +
                     self.G_COEFF * img_tensor[1] +
                     self.B_COEFF * img_tensor[2])
        luminance = luminance.unsqueeze(0)  # (1, H, W)

        return luminance.float(), torch.tensor(self.labels[idx], dtype=torch.float32)

---
## 3. CNN Model with Discriminative Gradient Features

In [ ]:

class CompactGradientNet(nn.Module):
    def __init__(self, depth=4, base_filters=32, dropout=0.1, embedding_dim=128):
        """
        CNN for gradient field classification with discriminative features.
        6-channel input (single scale): [Gx, Gy, magnitude, sin(angle), cos(angle), coherence]

        v3 improvements:
        - Single scale only (coarse blur was destroying high-freq artifacts)
        - Input BatchNorm to normalize across heterogeneous channels
        - Learnable 1x1 conv for channel weighting before main CNN
        - Lower dropout (0.1 default) to reduce underfitting
        """
        super().__init__()

        self.N_GRAD_CHANNELS = 6  # single-scale: Gx, Gy, mag, sin, cos, coh

        # Sobel kernels (fixed, not trainable)
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                               dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
                               dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

        # Gaussian kernel for structure tensor smoothing
        gaussian = torch.tensor([[1, 4, 6, 4, 1], [4, 16, 24, 16, 4],
                                  [6, 24, 36, 24, 6], [4, 16, 24, 16, 4],
                                  [1, 4, 6, 4, 1]], dtype=torch.float32) / 256.0
        self.register_buffer('gaussian', gaussian.view(1, 1, 5, 5))

        # --- NEW: Input normalization ---
        # Normalizes heterogeneous channel scales (Gx/Gy ~[-4,4], sin/cos [-1,1], coherence [0,1])
        self.input_norm = nn.BatchNorm2d(self.N_GRAD_CHANNELS)

        # --- NEW: Learnable 1x1 conv ---
        # Let the model learn optimal channel weighting/mixing before the main CNN
        self.channel_mix = nn.Sequential(
            nn.Conv2d(self.N_GRAD_CHANNELS, self.N_GRAD_CHANNELS, kernel_size=1),
            nn.ReLU(),
        )

        # CNN layers
        layers = []
        in_ch = self.N_GRAD_CHANNELS
        for i in range(depth):
            out_ch = base_filters * (2**i)
            layers.extend([
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
                nn.MaxPool2d(2)
            ])
            if dropout > 0:
                layers.append(nn.Dropout2d(dropout))
            in_ch = out_ch

        self.cnn = nn.Sequential(*layers)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.embedding = nn.Linear(out_ch, embedding_dim)
        self.classifier = nn.Linear(embedding_dim, 1)

    def compute_gradient_field_single_scale(self, luminance):
        """Compute 6-channel gradient field for a given luminance."""
        G_x = F.conv2d(luminance, self.sobel_x, padding=1)
        G_y = F.conv2d(luminance, self.sobel_y, padding=1)

        magnitude = torch.sqrt(G_x**2 + G_y**2 + 1e-8)
        angle = torch.atan2(G_y, G_x)

        sin_angle = torch.sin(angle)
        cos_angle = torch.cos(angle)

        # Structure tensor for coherence
        Gxx, Gxy, Gyy = G_x * G_x, G_x * G_y, G_y * G_y
        Sxx = F.conv2d(Gxx, self.gaussian, padding=2)
        Sxy = F.conv2d(Gxy, self.gaussian, padding=2)
        Syy = F.conv2d(Gyy, self.gaussian, padding=2)

        trace = Sxx + Syy
        det_term = torch.sqrt((Sxx - Syy)**2 + 4 * Sxy**2 + 1e-8)
        lambda1, lambda2 = 0.5 * (trace + det_term), 0.5 * (trace - det_term)
        coherence = ((lambda1 - lambda2) / (lambda1 + lambda2 + 1e-8))**2

        magnitude_scaled = torch.log1p(magnitude * 10)

        return torch.cat([G_x, G_y, magnitude_scaled, sin_angle, cos_angle, coherence], dim=1)

    def compute_gradient_field(self, luminance):
        """Compute 6-channel gradient field on GPU (single scale — no blurring)."""
        return self.compute_gradient_field_single_scale(luminance)

    def forward(self, luminance):
        x = self.compute_gradient_field(luminance)
        x = self.input_norm(x)       # normalize heterogeneous channels
        x = self.channel_mix(x)      # learnable 1x1 channel weighting
        x = self.cnn(x)
        x = self.global_pool(x).flatten(1)
        emb = self.embedding(x)
        logit = self.classifier(emb)
        return logit.squeeze(1), emb



In [ ]:

from tqdm import tqdm

def one_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch. Returns (mean_loss, train_accuracy)."""
    model.train()
    losses = []
    correct = 0
    total = 0
    for x, y in tqdm(loader, desc="Training", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits, _ = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        # Track training accuracy
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == y).sum().item()
        total += y.size(0)
    train_acc = correct / total if total > 0 else 0.0
    return np.mean(losses), train_acc


---
## 4. Training and Evaluation Functions

In [9]:
from tqdm import tqdm

def one_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    losses = []
    for x, y in tqdm(loader, desc="Training", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits, _ = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return np.mean(losses)



In [10]:
@torch.inference_mode()
def evaluate(model, loader, device, threshold=0.5) -> Metrics:
    """
    Evaluate model and return comprehensive Metrics object.

    Args:
        model: Trained model
        loader: DataLoader
        device: Device to use
        threshold: Classification threshold (default 0.5)

    Returns:
        Metrics object with all evaluation metrics
    """
    model.eval()
    all_labels, all_probs = [], []

    for x, y in tqdm(loader, desc="Evaluating", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits, _ = model(x)
        probs = torch.sigmoid(logits)
        all_labels.append(y.cpu())
        all_probs.append(probs.cpu())

    if len(all_labels) == 0:
        return Metrics()

    labels = torch.cat(all_labels).numpy()
    probs = torch.cat(all_probs).numpy().flatten()
    preds = (probs >= threshold).astype(int)

    return compute_metrics(labels, preds, probs)



---
## 5. Visualization Functions

In [11]:
def plot_metrics_curves(metrics: Metrics, title_prefix=""):
    """
    Plot ROC curve, Precision-Recall curve, and probability distribution.

    Args:
        metrics: Metrics object with labels, preds, probs
        title_prefix: Optional prefix for plot titles
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    labels, probs = metrics.labels, metrics.probs

    # 1. ROC Curve
    if not np.isnan(metrics.auroc):
        fpr, tpr, thresholds = roc_curve(labels, probs)
        axes[0].plot(fpr, tpr, 'b-', lw=2, label=f'ROC (AUC = {metrics.auroc:.3f})')
        axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
        axes[0].fill_between(fpr, tpr, alpha=0.2)

        # Find optimal threshold (Youden's J)
        j_scores = tpr - fpr
        best_idx = np.argmax(j_scores)
        best_thresh = thresholds[best_idx]
        axes[0].scatter(fpr[best_idx], tpr[best_idx], s=100, c='red',
                       label=f'Optimal (t={best_thresh:.2f})', zorder=5)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title(f'{title_prefix}ROC Curve')
    axes[0].legend(loc='lower right')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim([0, 1])
    axes[0].set_ylim([0, 1])

    # 2. Precision-Recall Curve
    if not np.isnan(metrics.avg_precision):
        precision, recall, _ = precision_recall_curve(labels, probs)
        axes[1].plot(recall, precision, 'g-', lw=2,
                    label=f'PR (AP = {metrics.avg_precision:.3f})')
        axes[1].fill_between(recall, precision, alpha=0.2, color='green')

        # Baseline (random classifier)
        baseline = labels.sum() / len(labels)
        axes[1].axhline(y=baseline, color='k', linestyle='--', label=f'Baseline ({baseline:.2f})')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title(f'{title_prefix}Precision-Recall Curve')
    axes[1].legend(loc='lower left')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([0, 1])
    axes[1].set_ylim([0, 1])

    # 3. Probability Distribution
    real_probs = probs[labels == 0]
    fake_probs = probs[labels == 1]

    axes[2].hist(real_probs, bins=50, alpha=0.6, color='green', label='Real', density=True)
    axes[2].hist(fake_probs, bins=50, alpha=0.6, color='red', label='Fake', density=True)
    axes[2].axvline(x=0.5, color='k', linestyle='--', lw=2, label='Threshold (0.5)')
    axes[2].set_xlabel('Predicted Probability (Fake)')
    axes[2].set_ylabel('Density')
    axes[2].set_title(f'{title_prefix}Probability Distribution')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [12]:
def plot_confusion_matrix(metrics: Metrics, title="Confusion Matrix"):
    """
    Plot confusion matrix with percentages.
    """
    cm = confusion_matrix(metrics.labels, metrics.preds)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Raw counts
    im = axes[0].imshow(cm, cmap='Blues')
    axes[0].set_title(f'{title} (Counts)')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].set_xticks([0, 1])
    axes[0].set_yticks([0, 1])
    axes[0].set_xticklabels(['Real', 'Fake'])
    axes[0].set_yticklabels(['Real', 'Fake'])
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, f'{cm[i, j]}', ha='center', va='center', fontsize=18,
                        color='white' if cm[i, j] > cm.max()/2 else 'black')
    plt.colorbar(im, ax=axes[0])

    # Percentages
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    im = axes[1].imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
    axes[1].set_title(f'{title} (Row-Normalized %)')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    axes[1].set_xticks([0, 1])
    axes[1].set_yticks([0, 1])
    axes[1].set_xticklabels(['Real', 'Fake'])
    axes[1].set_yticklabels(['Real', 'Fake'])
    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, f'{cm_pct[i, j]:.1f}%', ha='center', va='center', fontsize=18,
                        color='white' if cm_pct[i, j] > 50 else 'black')
    plt.colorbar(im, ax=axes[1])

    plt.tight_layout()
    plt.show()

    # Print summary
    tn, fp, fn, tp = cm.ravel()
    print(f"\n{'='*50}")
    print(f"CLASSIFICATION SUMMARY")
    print(f"{'='*50}")
    print(f"True Negatives:   {tn:5d}  ({tn/(tn+fp)*100:.1f}%)")
    print(f"False Positives:  {fp:5d}  ({fp/(tn+fp)*100:.1f}%)")
    print(f"False Negatives:  {fn:5d}  ({fn/(fn+tp)*100:.1f}%)")
    print(f"True Positives:   {tp:5d}  ({tp/(fn+tp)*100:.1f}%)")
    print(f"{'='*50}")

In [13]:
def plot_training_history(history: dict):
    """
    Plot training curves from history dictionary.

    Args:
        history: Dict with 'train_loss' and 'val_metrics' (list of Metrics)
    """
    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Loss
    axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # 2. Accuracy, F1
    accs = [m.accuracy for m in history['val_metrics']]
    f1s = [m.f1 for m in history['val_metrics']]
    axes[1].plot(epochs, accs, 'r-o', label='Accuracy', markersize=4)
    axes[1].plot(epochs, f1s, 'g-o', label='F1 Score', markersize=4)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Score')
    axes[1].set_title('Accuracy & F1')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 1])

    # 3. AUROC, Average Precision
    aurocs = [m.auroc for m in history['val_metrics']]
    aps = [m.avg_precision for m in history['val_metrics']]
    axes[2].plot(epochs, aurocs, 'm-o', label='AUROC', markersize=4)
    axes[2].plot(epochs, aps, 'c-o', label='Avg Precision', markersize=4)

    best_auroc = max([x for x in aurocs if not np.isnan(x)], default=0)
    best_idx = aurocs.index(best_auroc) + 1
    axes[2].axvline(x=best_idx, color='k', linestyle='--', alpha=0.5, label=f'Best (Epoch {best_idx})')

    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Score')
    axes[2].set_title(f'AUROC & AP (Best AUROC: {best_auroc:.3f})')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    axes[2].set_ylim([0, 1])

    plt.tight_layout()
    plt.show()

In [14]:
def print_metrics_summary(metrics: Metrics, title="Metrics Summary"):
    """Print a nicely formatted metrics summary."""
    print(f"\n{'='*60}")
    print(f"{title:^60}")
    print(f"{'='*60}")
    print(f"{'Metric':<20} {'Value':>15} {'Description':>20}")
    print(f"{'-'*60}")
    print(f"{'Accuracy':<20} {metrics.accuracy:>15.4f} {'(TP+TN)/Total':>20}")
    print(f"{'Precision':<20} {metrics.precision:>15.4f} {'TP/(TP+FP)':>20}")
    print(f"{'Recall (Sensitivity)':<20} {metrics.recall:>15.4f} {'TP/(TP+FN)':>20}")
    print(f"{'Specificity':<20} {metrics.specificity:>15.4f} {'TN/(TN+FP)':>20}")
    print(f"{'F1 Score':<20} {metrics.f1:>15.4f} {'2*P*R/(P+R)':>20}")
    print(f"{'AUROC':<20} {metrics.auroc:>15.4f} {'Area under ROC':>20}")
    print(f"{'Avg Precision':<20} {metrics.avg_precision:>15.4f} {'Area under PR':>20}")
    print(f"{'='*60}\n")

---
## 6. Data Loading

In [ ]:
# Copies all three datasets from Google Drive to Colab local disk,
# merges them into a single train/ and test/ directory with real/fake
# subfolders, then builds the LuminanceDataset + DataLoaders.

import os
import shutil
from pathlib import Path
from tqdm import tqdm
from PIL import ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

# -------- Source datasets on Google Drive --------
DRIVE_ROOT = Path(GDRIVE_DATA_DIR)

DATASETS = {
    "OpenFake":   DRIVE_ROOT / "OpenFake",
    "WildFake":   DRIVE_ROOT / "wildfake_2k_split",
    "DRAGON":     DRIVE_ROOT / "DRAGON",
}

# -------- Combined output on Colab local disk --------
COMBINED_ROOT = Path("/content/combined_dataset")
COMBINED_TRAIN = COMBINED_ROOT / "train"
COMBINED_TEST  = COMBINED_ROOT / "test"

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}

def copy_images(src_dir: Path, dst_dir: Path):
    """Copy all image files from src_dir to dst_dir, avoiding name collisions."""
    if not src_dir.exists():
        print(f"  ⚠️  Skipping (not found): {src_dir}")
        return 0
    dst_dir.mkdir(parents=True, exist_ok=True)
    count = 0
    for f in src_dir.iterdir():
        if f.is_file() and f.suffix.lower() in IMG_EXTS:
            dest = dst_dir / f.name
            if dest.exists():
                # Avoid collision by prefixing with parent dataset name
                dest = dst_dir / f"{f.stem}_{count}{f.suffix}"
            shutil.copy2(f, dest)
            count += 1
    return count

# -------- Build combined dataset --------
if COMBINED_ROOT.exists():
    print("Combined dataset already exists, reusing it.")
else:
    print("Building combined dataset on local disk...")
    COMBINED_ROOT.mkdir(parents=True, exist_ok=True)

    total = {"train_real": 0, "train_fake": 0, "test_real": 0, "test_fake": 0}

    for name, base in DATASETS.items():
        if not base.exists():
            print(f"❌ {name}: not found at {base}, skipping.")
            continue
        print(f"\n📦 {name} ({base})")

        # Train
        n = copy_images(base / "train" / "real", COMBINED_TRAIN / "real")
        total["train_real"] += n
        print(f"  train/real: {n} images")

        n = copy_images(base / "train" / "fake", COMBINED_TRAIN / "fake")
        total["train_fake"] += n
        print(f"  train/fake: {n} images")

        # Test
        n = copy_images(base / "test" / "real", COMBINED_TEST / "real")
        total["test_real"] += n
        print(f"  test/real:  {n} images")

        n = copy_images(base / "test" / "fake", COMBINED_TEST / "fake")
        total["test_fake"] += n
        print(f"  test/fake:  {n} images")

    print(f"\n{'='*50}")
    print(f"COMBINED DATASET SUMMARY")
    print(f"{'='*50}")
    print(f"Train: {total['train_real']} real + {total['train_fake']} fake = {total['train_real']+total['train_fake']} total")
    print(f"Test:  {total['test_real']} real + {total['test_fake']} fake = {total['test_real']+total['test_fake']} total")
    print(f"Location: {COMBINED_ROOT}")

# -------- Load with existing get_image_paths_and_labels --------
def get_image_paths_and_labels(folder):
    if not os.path.exists(folder):
        print("warning: dataset folder not found")
        return [], []
    paths, labels = [], []
    for label, subfolder in tqdm(enumerate(["real", "fake"])):
        subdir = os.path.join(folder, subfolder)
        if not os.path.exists(subdir):
            print(f"warning: subfolder not found: {subdir}")
            continue
        try:
            files = os.listdir(subdir)
            valid_files = [f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))]
            for fname in valid_files:
                paths.append(os.path.join(subdir, fname))
                labels.append(label)
        except OSError as e:
          print(f"error accessing {subdir}: {e}")
    return paths, labels

train_paths, train_labels = get_image_paths_and_labels(str(COMBINED_TRAIN))
test_paths, test_labels   = get_image_paths_and_labels(str(COMBINED_TEST))

train_dataset = LuminanceDataset(train_paths, train_labels)
test_dataset  = LuminanceDataset(test_paths, test_labels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f"\nTrain: {len(train_dataset)} samples ({sum(train_labels)} fake, {len(train_labels)-sum(train_labels)} real)")
print(f"Test:  {len(test_dataset)} samples ({sum(test_labels)} fake, {len(test_labels)-sum(test_labels)} real)")
print(f"Device: {device}")


In [ ]:
"""def get_image_paths_and_labels(folder):
    if not os.path.exists(folder):
        print("warning: dataset folder not found")
        return [], []
    paths, labels = [], []
    for label, subfolder in tqdm(enumerate(["real", "fake"])):
        subdir = os.path.join(folder, subfolder)
        if not os.path.exists(subdir):
            print(f"warning: subfolder not found: {subdir}")
            continue
        try:
            files = os.listdir(subdir)
            valid_files = [f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))]
            for fname in valid_files:
                paths.append(os.path.join(subdir, fname))
                labels.append(label)
        except OSError as e:
          print(f"error accessing {subdir}: {e}")
    return paths, labels

# Load data
train_paths, train_labels = get_image_paths_and_labels(f"{data_dir}/train")
test_paths, test_labels = get_image_paths_and_labels(f"{data_dir}/test")

train_dataset = LuminanceDataset(train_paths, train_labels)
test_dataset = LuminanceDataset(test_paths, test_labels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                         num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset)} samples ({sum(train_labels)} fake, {len(train_labels)-sum(train_labels)} real)")
print(f"Test:  {len(test_dataset)} samples ({sum(test_labels)} fake, {len(test_labels)-sum(test_labels)} real)")
print(f"Device: {device}")
"""

NameError: name 'data_dir' is not defined

---
## 7. Training Loop

In [ ]:

# Checkpoint path on Google Drive
CHECKPOINT_DIR = Path("/content/drive/MyDrive/DeepfakeDetectionModels")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "gradient_field_cnn_v3.pth"

def train_model(resume=True):
    """Main training loop with checkpoint resumption support."""
    best_auroc = -1.0
    history = {"train_loss": [], "val_metrics": []}
    patience_counter = 0
    start_epoch = 0

    # Load data
    train_paths, train_labels = get_image_paths_and_labels(f"{data_dir}/train")
    test_paths, test_labels = get_image_paths_and_labels(f"{data_dir}/test")

    train_dataset = LuminanceDataset(train_paths, train_labels)
    test_dataset = LuminanceDataset(test_paths, test_labels)

    # OPTIMIZED DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=True
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=True
    )

    print(f"Train: {len(train_dataset)} samples ({sum(train_labels)} fake, {len(train_labels)-sum(train_labels)} real)")
    print(f"Test:  {len(test_dataset)} samples ({sum(test_labels)} fake, {len(test_labels)-sum(test_labels)} real)")

    # Initialize model — v3: single-scale, input norm, 1x1 conv, dropout=0.1
    model = CompactGradientNet(depth=4, base_filters=32, dropout=0.1).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Features: 6-channel [Gx, Gy, magnitude, sin(angle), cos(angle), coherence] (single scale)")
    print("=" * 80)

    for epoch in range(start_epoch, epochs):
        t0 = time.time()
        tr_loss, tr_acc = one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, test_loader, device)
        dt = time.time() - t0

        history["train_loss"].append(tr_loss)
        history["val_metrics"].append(val_metrics)
        scheduler.step(val_metrics.auroc)

        print(
            f"Epoch {epoch+1:2d}/{epochs} | {dt:5.1f}s | "
            f"Loss: {tr_loss:.4f} | TrainAcc: {tr_acc:.4f} | "
            f"ValAcc: {val_metrics.accuracy:.4f} | "
            f"F1: {val_metrics.f1:.4f} | AUROC: {val_metrics.auroc:.4f} | "
            f"AP: {val_metrics.avg_precision:.4f}"
        )

        if not np.isnan(val_metrics.auroc) and val_metrics.auroc > best_auroc:
            best_auroc = val_metrics.auroc
            patience_counter = 0

            # Save checkpoint to Google Drive
            checkpoint = {
                "model": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "epoch": epoch + 1,
                "best_auroc": best_auroc
            }
            torch.save(checkpoint, CHECKPOINT_PATH)
            print(f"  ✅ Saved checkpoint (AUROC={best_auroc:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  ⏹️ Early stopping at epoch {epoch+1}")
                break

    # Load best model
    checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    print(f"\n✅ Training complete. Best AUROC: {best_auroc:.4f}")
    print(f"   Checkpoint saved at: {CHECKPOINT_PATH}")

    return model, history


# Run training (set resume=False to start fresh)
model, history = train_model(resume=True)


In [25]:
def load_checkpoint():
  """ load checkpoint without needing to train"""
  # checkpoint_path = Path("/content/drive/MyDrive/DeepfakeDetectionModels/gradient_field_cnn_v3.pth")
  checkpoint_path = Path("/content/drive/MyDrive/DeepfakeDetectionModels/gradient_field_cnn_v3.pth")
  if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found at {checkpoint_path}")
  print(f"Loading checkpoint from {checkpoint_path}")
  checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
  model = CompactGradientNet(depth=4, base_filters=32, dropout=0.3, embedding_dim=256).to(device)
  # model = LargeGradientNet(depth=5, base_filters=64, dropout=0.4).to(device) <- large one
  if 'model' in checkpoint:
    print("found key: model")
    model.load_state_dict(checkpoint['model'])
  else:
    raise KeyError("Checkpoint does not contain a 'model' key")
  model.eval()

  epoch = checkpoint.get('epoch', 'Unknown')
  score = checkpoint.get('best_auroc', 'Unknown')
  print(f"✅ Model loaded (Epoch: {epoch}, Best AUROC: {score})")

  return model

model = load_checkpoint()


Loading checkpoint from ../models/grad_field_cnn/gradient_field_cnn_v2.pth
found key: model
✅ Model loaded (Epoch: 29, Best AUROC: 0.7762628448676638)


---
## 8. Results Visualization

In [20]:
# Diagnostic Analysis: Bias vs Variance
def analyze_training_bottlenecks(history):
    """
    Diagnose Bias vs Variance from learning curves.

    Args:
        history: Dict with 'train_loss' and 'val_metrics' (list of Metrics)
    """
    train_loss = history['train_loss']
    val_aurocs = [m.auroc for m in history['val_metrics']]
    val_accs = [m.accuracy for m in history['val_metrics']]
    epochs = range(1, len(train_loss) + 1)

    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Plot Train Loss (Primary Axis)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Training Loss', color='tab:blue')
    ax1.plot(epochs, train_loss, 'tab:blue', label='Train Loss', lw=2)
    ax1.tick_params(axis='y', labelcolor='tab:blue')
    ax1.grid(True, alpha=0.3)

    # Plot Val AUROC (Secondary Axis)
    ax2 = ax1.twinx()
    ax2.set_ylabel('Validation AUROC', color='tab:orange')
    ax2.plot(epochs, val_aurocs, 'tab:orange', label='Val AUROC', lw=2)
    ax2.tick_params(axis='y', labelcolor='tab:orange')
    ax2.set_ylim([0.5, 1.0])

    plt.title('Diagnostic Learning Curve: Bias vs Variance Check')

    # Interpretation Text
    final_loss = train_loss[-1]
    final_auroc = val_aurocs[-1]

    diagnosis = "Unknown"
    if final_loss > 0.4:
        diagnosis = "HIGH BIAS (Underfitting): Model might be too small/simple."
    elif final_loss < 0.2 and final_auroc < 0.8:
        diagnosis = "HIGH VARIANCE (Overfitting): Gap between Train/Val is large."
    else:
        diagnosis = "BALANCED: Performance might be saturated."

    plt.figtext(0.5, -0.05, f"Diagnosis Hint: {diagnosis}", ha="center", fontsize=12, bbox={"facecolor":"orange", "alpha":0.2, "pad":5})

    fig.tight_layout()
    plt.show()

# Run diagnostics on history
if 'history' in locals():
    analyze_training_bottlenecks(history)

In [21]:
# Training curves
plot_training_history(history)

NameError: name 'history' is not defined

---
## 10. Cross-Dataset Evaluation

In [22]:
# Dataset paths configuration
DATASETS = {
    "OpenFake": f"{GDRIVE_DATA_DIR}/OpenFake",
    "WildFake": f"{GDRIVE_DATA_DIR}/wildfake_2k_split",
    "DRAGON": f"{GDRIVE_DATA_DIR}/DRAGON"
}

# Verify which datasets are available
available_datasets = {}
for name, path in DATASETS.items():
    test_path = os.path.join(path, "test")
    if os.path.exists(test_path):
        available_datasets[name] = path
        print(f"✅ {name}: {test_path}")
    else:
        print(f"❌ {name}: Not found at {test_path}")

print(f"\nAvailable datasets: {list(available_datasets.keys())}")

NameError: name 'GDRIVE_DATA_DIR' is not defined

In [ ]:
# Evaluate on all available datasets
dataset_metrics = {}

for name, path in available_datasets.items():
    print(f"\n{"="*60}")
    print(f"Evaluating on: {name}")
    print(f"{"="*60}")

    # Load test data
    test_paths, test_labels = get_image_paths_and_labels(f"{path}/test")
    print(f"Samples: {len(test_paths)} ({sum(test_labels)} fake, {len(test_labels)-sum(test_labels)} real)")

    test_dataset = LuminanceDataset(test_paths, test_labels)
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    # Evaluate
    metrics = evaluate(model, test_loader, device)
    dataset_metrics[name] = metrics

    # Print summary
    print_metrics_summary(metrics, f"{name} Test Set Metrics")

print(f"\n✅ Evaluation complete on {len(dataset_metrics)} datasets")

In [ ]:
# Comparative ROC Curves
import pandas as pd

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. ROC Curves Comparison
colors = ["#2ecc71", "#3498db", "#e74c3c", "#9b59b6", "#f39c12"]
for idx, (name, metrics) in enumerate(dataset_metrics.items()):
    if not np.isnan(metrics.auroc):
        fpr, tpr, _ = roc_curve(metrics.labels, metrics.probs)
        axes[0].plot(fpr, tpr, color=colors[idx], lw=2,
                    label=f"{name} (AUC={metrics.auroc:.3f})")

axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="Random")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curves - Cross-Dataset Comparison")
axes[0].legend(loc="lower right")
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1])

# 2. PR Curves Comparison
for idx, (name, metrics) in enumerate(dataset_metrics.items()):
    if not np.isnan(metrics.avg_precision):
        precision, recall_vals, _ = precision_recall_curve(metrics.labels, metrics.probs)
        axes[1].plot(recall_vals, precision, color=colors[idx], lw=2,
                    label=f"{name} (AP={metrics.avg_precision:.3f})")

axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("PR Curves - Cross-Dataset Comparison")
axes[1].legend(loc="lower left")
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])

# 3. Metrics Bar Chart
metric_names = ["Accuracy", "F1", "AUROC"]
x = np.arange(len(metric_names))
width = 0.25

for idx, (name, metrics) in enumerate(dataset_metrics.items()):
    values = [metrics.accuracy, metrics.f1, metrics.auroc]
    axes[2].bar(x + idx*width, values, width, label=name, color=colors[idx])

axes[2].set_ylabel("Score")
axes[2].set_title("Metrics Comparison")
axes[2].set_xticks(x + width)
axes[2].set_xticklabels(metric_names)
axes[2].legend()
axes[2].set_ylim([0, 1])
axes[2].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "cross_dataset_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Confusion Matrices for each dataset
n_datasets = len(dataset_metrics)
fig, axes = plt.subplots(1, n_datasets, figsize=(6*n_datasets, 5))

if n_datasets == 1:
    axes = [axes]

for idx, (name, metrics) in enumerate(dataset_metrics.items()):
    cm = confusion_matrix(metrics.labels, metrics.preds)
    cm_pct = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis] * 100

    im = axes[idx].imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)
    axes[idx].set_title(f"{name}\n(Acc={metrics.accuracy:.1%})")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")
    axes[idx].set_xticks([0, 1])
    axes[idx].set_yticks([0, 1])
    axes[idx].set_xticklabels(["Real", "Fake"])
    axes[idx].set_yticklabels(["Real", "Fake"])

    for i in range(2):
        for j in range(2):
            axes[idx].text(j, i, f"{cm_pct[i,j]:.1f}%\n({cm[i,j]})",
                          ha="center", va="center", fontsize=12,
                          color="white" if cm_pct[i,j] > 50 else "black")

plt.suptitle("Confusion Matrices - Cross-Dataset Evaluation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "cross_dataset_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Summary Table
summary_data = []
for name, metrics in dataset_metrics.items():
    summary_data.append({
        "Dataset": name,
        "AUC": f"{metrics.auroc:.4f}",
        "F1": f"{metrics.f1:.4f}",
        "Accuracy": f"{metrics.accuracy:.4f}",
        "Precision": f"{metrics.precision:.4f}",
        "Recall": f"{metrics.recall:.4f}",
        "Specificity": f"{metrics.specificity:.4f}"
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("CROSS-DATASET EVALUATION SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

# Save to CSV
summary_df.to_csv(ARTIFACTS_DIR / "cross_dataset_metrics.csv", index=False)
print(f"\n✅ Results saved to {ARTIFACTS_DIR / "cross_dataset_metrics.csv"}")

In [ ]:
# Domain Gap Analysis
print("\n" + "="*80)
print("DOMAIN GAP ANALYSIS")
print("="*80)

if len(dataset_metrics) > 1:
    # Get reference dataset
    ref_name = list(dataset_metrics.keys())[0]
    ref_metrics = dataset_metrics[ref_name]

    print(f"\nReference: {ref_name} (AUC={ref_metrics.auroc:.4f})\n")

    for name, metrics in dataset_metrics.items():
        if name == ref_name:
            continue

        auc_gap = metrics.auroc - ref_metrics.auroc
        f1_gap = metrics.f1 - ref_metrics.f1
        acc_gap = metrics.accuracy - ref_metrics.accuracy

        status = "🟢" if auc_gap >= 0 else "🔴"
        print(f"{status} {name}:")
        print(f"   AUC: {metrics.auroc:.4f} ({auc_gap:+.4f})")
        print(f"   F1:  {metrics.f1:.4f} ({f1_gap:+.4f})")
        print(f"   Acc: {metrics.accuracy:.4f} ({acc_gap:+.4f})")
        print()
else:
    print("Only one dataset available - no domain gap analysis possible.")

# Probability distribution comparison
fig, axes = plt.subplots(1, len(dataset_metrics), figsize=(6*len(dataset_metrics), 4))
if len(dataset_metrics) == 1:
    axes = [axes]

for idx, (name, metrics) in enumerate(dataset_metrics.items()):
    real_probs = metrics.probs[metrics.labels == 0]
    fake_probs = metrics.probs[metrics.labels == 1]

    axes[idx].hist(real_probs, bins=50, alpha=0.6, color="green", label="Real", density=True)
    axes[idx].hist(fake_probs, bins=50, alpha=0.6, color="red", label="Fake", density=True)
    axes[idx].axvline(x=0.5, color="k", linestyle="--", lw=2)
    axes[idx].set_xlabel("P(Fake)")
    axes[idx].set_ylabel("Density")
    axes[idx].set_title(f"{name} - Probability Distribution")
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.suptitle("Prediction Confidence by Dataset", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "cross_dataset_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 11. Export for Fusion

Save model artifacts for fusion-layer compatibility.

In [26]:
# Export model artifacts for fusion
import json
from pathlib import Path

EXPORT_DIR = Path("../models/grad_field_cnn")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save weights
weights_path = EXPORT_DIR / "weights.pt"
torch.save({"model_state": model.state_dict()}, weights_path)
print(f"✅ Saved weights to {weights_path}")

# 2. Export config with preprocessing params
config = {
    "model": {
        "name": "CompactGradientNet",
        "version": "v2",
        "depth": 4,
        "base_filters": 32,
        "dropout": 0.3,
        "embedding_dim": 128
    },
    "preprocessing": {
        "image_size": 224,
        "luminance": {"standard": "BT.709", "R": 0.2126, "G": 0.7152, "B": 0.0722},
        "gradient": {
            "operator": "Sobel",
            "kernel_size": 3,
            "sobel_x": [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
            "sobel_y": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]]
        },
        "structure_tensor": {"gaussian_kernel_size": 5},
        "features": {
            "channels": ["Gx", "Gy", "magnitude_scaled", "angle_normalized", "coherence"],
            "magnitude_transform": "log1p(magnitude * 10)",
            "angle_transform": "atan2(Gy, Gx) / pi",
            "coherence_formula": "((lambda1 - lambda2) / (lambda1 + lambda2 + eps))^2"
        },
        "epsilon": 1e-8
    },
    "input_spec": {"channels": 1, "height": 256, "width": 256, "dtype": "float32"},
    "output_spec": {
        "logit": {"shape": [1], "description": "positive = fake"},
        "embedding": {"shape": [128], "description": "for fusion"}
    },
    "training": {
        "dataset": "OpenFake",
        "epochs": epochs,
        "batch_size": batch_size,
        "learning_rate": lr
    }
}

config_path = EXPORT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"✅ Saved config to {config_path}")

# Summary
print(f"\n📦 Export complete!")
print(f"   {EXPORT_DIR}/weights.pt")
print(f"   {EXPORT_DIR}/config.json")
print(f"   {EXPORT_DIR}/predict.py (already exists)")

✅ Saved weights to ../models/grad_field_cnn/weights.pt
✅ Saved config to ../models/grad_field_cnn/config.json

📦 Export complete!
   ../models/grad_field_cnn/weights.pt
   ../models/grad_field_cnn/config.json
   ../models/grad_field_cnn/predict.py (already exists)


---
## 9. Gradient Feature Visualization

In [ ]:
def visualize_gradient_features(dataset, model, device, index):
    """Visualize 6-channel gradient field for a sample."""
    model.eval()
    label = "REAL" if dataset.labels[index] == 0 else "FAKE"

    img = Image.open(dataset.img_paths[index]).convert('RGB')
    img = transforms.Resize((256, 256))(img)

    luminance, _ = dataset[index]
    with torch.no_grad():
        gf = model.compute_gradient_field(luminance.unsqueeze(0).to(device))[0].cpu().numpy()

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    axes[0,0].imshow(np.array(img))
    axes[0,0].set_title('Original')
    axes[0,0].axis('off')

    axes[0,1].imshow(luminance[0], cmap='gray')
    axes[0,1].set_title('Luminance')
    axes[0,1].axis('off')

    for i, (name, cmap) in enumerate([('Gx', 'RdBu'), ('Gy', 'RdBu'),
                                       ('Magnitude', 'hot'), ('Angle', 'hsv'), ('Coherence', 'viridis')]):
        ax = axes[(i+2)//4, (i+2)%4]
        im = ax.imshow(gf[i], cmap=cmap)
        ax.set_title(name)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)

    axes[1,3].axis('off')
    plt.suptitle(f'{label} Sample - Gradient Features', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Visualize real and fake samples
real_idx = next(i for i, l in enumerate(test_dataset.labels) if l == 0)
fake_idx = next(i for i, l in enumerate(test_dataset.labels) if l == 1)

visualize_gradient_features(test_dataset, model, device, real_idx)
visualize_gradient_features(test_dataset, model, device, fake_idx)

NameError: name 'test_dataset' is not defined

---
# Gradient Field Settings

In [28]:
GRADFIELD_SETTINGS = {
    # how gradients are generated:
    "operator": "sobel",             # "sobel" | "scharr" | "prewitt" | "custom"
    "kernel_size": 3,
    "use_magnitude": True,           # True => sqrt(gx^2 + gy^2)
    "use_direction": True,           # True => atan2(gy, gx)
    "use_gx_gy_channels": True,      # True => keep gx and gy separately
    "use_coherence": True,           # True => structure tensor coherence

    # how gradients are combined with image:
    "input_mode": "grad_only",       # "grad_only" | "rgb_plus_grad"
    "rgb_plus_grad_concat_order": "rgb_first",  # if used

    # scaling / normalization:
    "grad_scale": "minmax_0_1",      # "minmax_0_1" | "standardize" | "none"
    "epsilon": 1e-6,

    # channel expectations:
    "expected_channels": 5,          # Gx, Gy, Magnitude, Angle, Coherence
    "channel_names": ["Gx", "Gy", "magnitude", "angle", "coherence"]
}

# Export artifacts (weights + configs)

In [ ]:
import os, json, torch

EXPORT_DIR = "../models/grad_field_cnn"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Save weights (state_dict)
torch.save(model.state_dict(), os.path.join(EXPORT_DIR, "model.pth"))
print(f"✅ Saved model weights to {EXPORT_DIR}/model.pth")

# Preprocessing details (MUST match training)
# NOTE: normalize here refers to what you apply to the model input AFTER gradient-field generation.
preprocess = {
    "input_size": 256,
    "center_crop": True,
    "normalize": {
        "mean": [0.5] * GRADFIELD_SETTINGS["expected_channels"],
        "std":  [0.5] * GRADFIELD_SETTINGS["expected_channels"]
    }
}

label_map = {"0": "real", "1": "fake"}

config = {
    "name": "gradfield-cnn",
    "framework": "pytorch",
    "arch": "CompactGradientNet",
    "version": "v2",
    "num_classes": 2,
    "threshold": 0.50,
    "labels": label_map,
    "model_parameters": {
        "depth": 4,
        "base_filters": 32,
        "dropout": 0.3,
        "embedding_dim": 128
    },

    # CRITICAL: gradient-field generation parameters
    "gradfield_settings": GRADFIELD_SETTINGS,

    "notes": "Gradient Field CNN baseline for deepfake detection"
}

with open(os.path.join(EXPORT_DIR, "preprocess.json"), "w") as f:
    json.dump(preprocess, f, indent=2)

with open(os.path.join(EXPORT_DIR, "label_map.json"), "w") as f:
    json.dump(label_map, f, indent=2)

with open(os.path.join(EXPORT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

readme = """---
license: apache-2.0
tags:
  - deepfake-detection
  - image-classification
  - cnn
  - gradients
---

# Gradient Field CNN – DeepFakeDetector

CNN trained on gradient-field representations (Gx, Gy, Magnitude, Angle, Coherence) for binary deepfake detection.

## Labels
- 0 = real
- 1 = fake

## Gradient Field Settings
This repo includes `gradfield_settings` inside `config.json` which defines:
- gradient operator (sobel)
- channel construction (6 channels)
- scaling/normalization
"""

with open(os.path.join(EXPORT_DIR, "README.md"), "w") as f:
    f.write(readme)

print("Exported Gradient Field CNN artifacts to:", EXPORT_DIR)
print("gradfield_settings:", GRADFIELD_SETTINGS)

✅ Saved model weights to ../models/grad_field_cnn/model.pth
Exported Gradient Field CNN artifacts to: ../models/grad_field_cnn
gradfield_settings: {'operator': 'sobel', 'kernel_size': 3, 'use_magnitude': True, 'use_direction': True, 'use_gx_gy_channels': True, 'use_coherence': True, 'input_mode': 'grad_only', 'rgb_plus_grad_concat_order': 'rgb_first', 'grad_scale': 'minmax_0_1', 'epsilon': 1e-06, 'expected_channels': 5, 'channel_names': ['Gx', 'Gy', 'magnitude', 'angle', 'coherence']}


# Upload artifacts to HuggingFace

In [30]:
# Artifacts are exported locally to models/grad_field_cnn. Manual upload can be done later.

# sanity reload test


In [31]:
# Verifying Export
print("\n--- Verifying Exported Model ---")
try:
    # Load config
    with open(os.path.join(EXPORT_DIR, "config.json"), "r") as f:
        loaded_config = json.load(f)
    
    # Initialize model from config
    model_params = loaded_config["model_parameters"]
    loaded_model = CompactGradientNet(**model_params).to(device)
    
    # Load weights
    loaded_model.load_state_dict(torch.load(os.path.join(EXPORT_DIR, "model.pth"), map_location=device))
    loaded_model.eval()
    print("✅ Model loaded successfully from export")
    
    # Compare settings
    exported_settings = loaded_config["gradfield_settings"]
    assert exported_settings["expected_channels"] == 5, "Mismatch in expected channels"
    print("✅ Config settings verified")
    
except Exception as e:
    print(f"❌ Verification failed: {e}")


--- Verifying Exported Model ---
✅ Model loaded successfully from export
✅ Config settings verified


In [ ]:
# ==========================================
# CELL: Large-Sample Coherence Distribution Plots + Confidence Intervals
# Add after Section 9 (Gradient Feature Visualization) in gradient_field_cnn_baseline.ipynb
#
# Computes per-image coherence statistics across the full combined dataset,
# then plots real vs fake distributions with 95% bootstrap CIs.
# ==========================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from tqdm import tqdm
import torch

@torch.inference_mode()
def extract_coherence_stats(model, dataset, device, max_samples=None):
    """
    Extract per-image coherence statistics from the model's gradient field.

    For each image, computes:
      - mean coherence
      - median coherence
      - std of coherence
      - 90th percentile (high-coherence fraction)

    Returns dict with arrays keyed by stat name, plus 'labels'.
    """
    model.eval()

    n = len(dataset) if max_samples is None else min(max_samples, len(dataset))
    means, medians, stds, p90s, labels = [], [], [], [], []

    for i in tqdm(range(n), desc="Extracting coherence"):
        lum, label = dataset[i]
        lum = lum.unsqueeze(0).to(device)  # (1, 1, H, W)

        # Compute gradient field (12 channels: 2 scales x 6 features)
        gf = model.compute_gradient_field(lum)  # (1, 12, H, W)

        # Coherence is channel index 5 (fine scale) and 11 (coarse scale)
        # [Gx, Gy, magnitude, sin(angle), cos(angle), coherence] x 2 scales
        coh_fine   = gf[0, 5].cpu().numpy()   # fine-scale coherence
        coh_coarse = gf[0, 11].cpu().numpy()  # coarse-scale coherence

        # Average coherence across both scales
        coh = (coh_fine + coh_coarse) / 2.0

        means.append(float(coh.mean()))
        medians.append(float(np.median(coh)))
        stds.append(float(coh.std()))
        p90s.append(float(np.percentile(coh, 90)))
        labels.append(int(label))

    return {
        "mean": np.array(means),
        "median": np.array(medians),
        "std": np.array(stds),
        "p90": np.array(p90s),
        "labels": np.array(labels),
    }


def bootstrap_ci(data, n_boot=2000, ci=0.95, stat_fn=np.mean):
    """Compute bootstrap confidence interval for a statistic."""
    rng = np.random.default_rng(42)
    boot_stats = np.array([
        stat_fn(rng.choice(data, size=len(data), replace=True))
        for _ in range(n_boot)
    ])
    alpha = (1 - ci) / 2
    lo, hi = np.percentile(boot_stats, [100 * alpha, 100 * (1 - alpha)])
    return stat_fn(data), lo, hi


def plot_coherence_distributions(coh_stats, save_path=None):
    """
    Plot coherence distributions for real vs fake images with CIs.

    4-panel figure:
      1. Histogram of mean coherence (real vs fake)
      2. Histogram of coherence std (texture uniformity)
      3. Box + strip plot of mean coherence by class
      4. Bar chart of mean ± 95% CI for each statistic
    """
    labels = coh_stats["labels"]
    real_mask = labels == 0
    fake_mask = labels == 1

    n_real = int(real_mask.sum())
    n_fake = int(fake_mask.sum())

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # ===== Panel 1: Mean Coherence Distribution =====
    ax = axes[0, 0]
    bins = np.linspace(0, 1, 60)
    ax.hist(coh_stats["mean"][real_mask], bins=bins, alpha=0.6, color="#2ecc71",
            label=f"Real (n={n_real})", density=True, edgecolor="white", linewidth=0.5)
    ax.hist(coh_stats["mean"][fake_mask], bins=bins, alpha=0.6, color="#e74c3c",
            label=f"Fake (n={n_fake})", density=True, edgecolor="white", linewidth=0.5)

    # Add means with CI
    for mask, color, lbl in [(real_mask, "#27ae60", "Real"), (fake_mask, "#c0392b", "Fake")]:
        mu, lo, hi = bootstrap_ci(coh_stats["mean"][mask])
        ax.axvline(mu, color=color, linestyle="--", lw=2)
        ax.axvspan(lo, hi, alpha=0.15, color=color)
        ax.text(mu, ax.get_ylim()[1]*0.9, f"{lbl} μ={mu:.4f}\n[{lo:.4f}, {hi:.4f}]",
                ha="center", fontsize=8, color=color, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

    ax.set_xlabel("Mean Coherence per Image")
    ax.set_ylabel("Density")
    ax.set_title("Mean Coherence Distribution")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)

    # ===== Panel 2: Coherence Std (Texture Uniformity) =====
    ax = axes[0, 1]
    bins_std = np.linspace(0, 0.5, 50)
    ax.hist(coh_stats["std"][real_mask], bins=bins_std, alpha=0.6, color="#2ecc71",
            label="Real", density=True, edgecolor="white", linewidth=0.5)
    ax.hist(coh_stats["std"][fake_mask], bins=bins_std, alpha=0.6, color="#e74c3c",
            label="Fake", density=True, edgecolor="white", linewidth=0.5)

    for mask, color in [(real_mask, "#27ae60"), (fake_mask, "#c0392b")]:
        mu, lo, hi = bootstrap_ci(coh_stats["std"][mask])
        ax.axvline(mu, color=color, linestyle="--", lw=2)
        ax.axvspan(lo, hi, alpha=0.15, color=color)

    ax.set_xlabel("Coherence Std per Image")
    ax.set_ylabel("Density")
    ax.set_title("Coherence Spatial Variability\n(lower std → more uniform texture)")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)

    # ===== Panel 3: Box + Strip Plot =====
    ax = axes[1, 0]
    data_box = [coh_stats["mean"][real_mask], coh_stats["mean"][fake_mask]]
    bp = ax.boxplot(data_box, labels=["Real", "Fake"], patch_artist=True,
                    widths=0.5, showmeans=True,
                    meanprops=dict(marker="D", markerfacecolor="gold", markersize=8))
    colors_box = ["#2ecc71", "#e74c3c"]
    for patch, c in zip(bp["boxes"], colors_box):
        patch.set_facecolor(c)
        patch.set_alpha(0.4)

    # Overlay individual points (jittered)
    rng = np.random.default_rng(0)
    for i, (d, c) in enumerate(zip(data_box, colors_box)):
        jitter = rng.normal(0, 0.04, size=len(d))
        ax.scatter(np.full_like(d, i + 1) + jitter, d, alpha=0.15, s=8, color=c, zorder=2)

    ax.set_ylabel("Mean Coherence")
    ax.set_title("Mean Coherence by Class")
    ax.grid(True, alpha=0.3, axis="y")

    # Statistical test
    t_stat, p_val = stats.mannwhitneyu(
        coh_stats["mean"][real_mask], coh_stats["mean"][fake_mask], alternative="two-sided"
    )
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
    ax.set_xlabel(f"Mann-Whitney U p={p_val:.2e} ({sig})")

    # ===== Panel 4: Summary Bar Chart with 95% CIs =====
    ax = axes[1, 1]
    stat_names = ["Mean Coh.", "Median Coh.", "Coh. Std", "90th Pctl"]
    stat_keys  = ["mean", "median", "std", "p90"]

    x = np.arange(len(stat_names))
    width = 0.35

    for offset, mask, color, lbl in [
        (-width/2, real_mask, "#2ecc71", "Real"),
        (+width/2, fake_mask, "#e74c3c", "Fake"),
    ]:
        centers, lowers, uppers = [], [], []
        for key in stat_keys:
            mu, lo, hi = bootstrap_ci(coh_stats[key][mask])
            centers.append(mu)
            lowers.append(mu - lo)
            uppers.append(hi - mu)

        ax.bar(x + offset, centers, width, label=lbl, color=color, alpha=0.7,
               edgecolor="white", linewidth=0.8)
        ax.errorbar(x + offset, centers, yerr=[lowers, uppers],
                    fmt="none", ecolor="black", capsize=4, capthick=1.5, lw=1.5)

    ax.set_xticks(x)
    ax.set_xticklabels(stat_names)
    ax.set_ylabel("Value")
    ax.set_title("Coherence Statistics with 95% Bootstrap CI")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

    plt.suptitle("Coherence Distribution Analysis: Real vs Fake",
                 fontsize=16, fontweight="bold", y=1.01)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"✅ Saved to {save_path}")

    plt.show()

    # Print numerical summary
    print(f"\n{'='*70}")
    print(f"{'COHERENCE STATISTICS SUMMARY':^70}")
    print(f"{'='*70}")
    print(f"{'Statistic':<20} {'Real (mean [95% CI])':<25} {'Fake (mean [95% CI])':<25}")
    print(f"{'-'*70}")
    for key, name in zip(["mean", "median", "std", "p90"],
                         ["Mean Coh.", "Median Coh.", "Coh. Std", "90th Pctl"]):
        r_mu, r_lo, r_hi = bootstrap_ci(coh_stats[key][real_mask])
        f_mu, f_lo, f_hi = bootstrap_ci(coh_stats[key][fake_mask])
        print(f"{name:<20} {r_mu:.4f} [{r_lo:.4f}, {r_hi:.4f}]    {f_mu:.4f} [{f_lo:.4f}, {f_hi:.4f}]")
    print(f"{'='*70}")
    print(f"Samples: {n_real} real, {n_fake} fake")
    print(f"Mann-Whitney U test on mean coherence: p = {p_val:.2e}")


# -------- Run the analysis --------
# Uses the model and test_dataset already defined in the notebook
print("Extracting coherence statistics from test set...")
coh_stats = extract_coherence_stats(model, test_dataset, device)

plot_coherence_distributions(
    coh_stats,
    save_path=ARTIFACTS_DIR / "coherence_distribution_analysis.png"
)
